# Hypothesis Testing & Customer Segmentation
Task 4: Data Storytelling & Statistical Validation

## Step 1: Load Data & Check Columns

In [ ]:
import pandas as pd

df = pd.read_csv("data/cleaned_sales_dataset.csv")
print(df.columns.tolist())
df.head()

## Step 2: Customer Segmentation (High / Medium / Low Value)

**Note:** If your column is not named `Total_Sales`, change it below to match the output from Step 1.

In [ ]:
SALES_COLUMN = "Total_Sales"  # <-- change this if your column name is different

df['Customer_Segment'] = pd.qcut(
    df[SALES_COLUMN],
    q=[0, 0.5, 0.8, 1.0],
    labels=['Low Value', 'Medium Value', 'High Value']
)

segment_revenue = df.groupby('Customer_Segment')[SALES_COLUMN].sum()
segment_revenue_pct = (segment_revenue / segment_revenue.sum() * 100).round(1)

print("Revenue Share by Segment (%):")
print(segment_revenue_pct)

## Step 3: Hypothesis Testing (T-test)

- **H0 (Null):** No significant difference in AOV between High Value and other customers.
- **H1 (Alternative):** High Value customers have a significantly higher AOV than other customers.

In [ ]:
from scipy import stats

high_value_aov = df[df['Customer_Segment'] == 'High Value'][SALES_COLUMN]
other_aov = df[df['Customer_Segment'] != 'High Value'][SALES_COLUMN]

t_stat, p_value = stats.ttest_ind(high_value_aov, other_aov, equal_var=False)

diff_mean = high_value_aov.mean() - other_aov.mean()
se = ((high_value_aov.std()**2 / len(high_value_aov)) +
      (other_aov.std()**2 / len(other_aov))) ** 0.5
ci_low = diff_mean - 1.96 * se
ci_high = diff_mean + 1.96 * se

print(f"High Value mean AOV: {high_value_aov.mean():.2f}")
print(f"Other segments mean AOV: {other_aov.mean():.2f}")
print(f"T-statistic: {t_stat:.2f}")
print(f"P-value: {p_value:.5f}")
print(f"95% CI for difference: [{ci_low:.2f}, {ci_high:.2f}]")

## Step 4: Interpretation & Conclusion

In [ ]:
alpha = 0.05
if p_value < alpha:
    print("Result: Reject H0 — the difference is statistically significant.")
    print("Business takeaway: High Value customers genuinely spend more per order;\n"
          "this segment is worth a dedicated retention/loyalty strategy.")
else:
    print("Result: Fail to reject H0 — no statistically significant difference found.")